# N-LDF benchmark on 41 KEEL tasks — re-runnable notebook

This notebook re-runs the benchmark reported in the manuscript with **exactly
the sampler implementation that produced the archived results**: the class in
`nldf.py` (SHA-256 `77c5f727…9872`, verified in Cell 3) and the sixteen
parameter values fixed by `nldf.build_sampler`. Everything else (data loading,
fold-wise preprocessing, the comparison oversamplers at imbalanced-learn
defaults, the four classifier policies, the 5×5 schedule with its prespecified
reductions, and the post-processing) is the executed harness, cell for cell.

It differs from the executed notebooks kept in `notebooks/` only in what makes
a re-run unambiguous:

* the sampler is imported from `nldf.py` instead of being defined (twice) inline;
* there is no interactive prompt; `CLEAN_START` and `SMOKE_TEST` are constants;
* the environment report reads the parameter values from the sampler object
  instead of a retyped dictionary, and no longer lists inner cross-validation
  as a component of the benchmark (the option exists in the class and was
  disabled: `use_inner_cv=False`, sampling ratio fixed at 0.75);
* the critical-difference figure cells are omitted (the paper does not use a CD
  diagram; `make_fig_results.py` draws the figures from the archived records).

A full run takes about three hours on a Colab CPU runtime. Exact numerical
reproduction across library versions is not guaranteed; the archived run used
Python 3.12.13, numpy 2.0.2, pandas 2.2.2, scipy 1.16.3, scikit-learn 1.6.1,
imbalanced-learn 0.14.2 and xgboost 3.3.0.


### Cell 0: packages and code

In [ ]:
"""Cell 0: packages and repository code.

Run from the root of the n-ldf repository (locally), or let the cell clone it
(Colab). Nothing here mounts Google Drive; results go to ./nldf_results_41_v2_final,
the output directory name of the archived run.
"""
import os
import subprocess
import sys
from pathlib import Path

for pkg in ["imbalanced-learn", "xgboost", "scikit-posthocs", "requests", "tqdm"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

if not Path("nldf.py").exists():
    subprocess.check_call(["git", "clone", "-q", "https://github.com/ahmadilham-idsrg/n-ldf.git"])
    os.chdir("n-ldf")
print("working directory:", Path.cwd())
print("nldf.py present   :", Path("nldf.py").exists())


### Cell 1: imports, constants and the 41 KEEL tasks

In [ ]:
"""Cell 1 (REVISED): Import dan Constants dengan KDD Cup Fix"""

from __future__ import annotations

import argparse
import hashlib
import importlib.metadata
import io
import json
import pickle
import platform
import re
import signal
import sys
import time
import zipfile
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Optional

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from tqdm import tqdm
from urllib3.util.retry import Retry

warnings.filterwarnings('ignore')

# Optional dependencies
try:
    from imblearn.combine import SMOTETomek
    from imblearn.over_sampling import ADASYN, BorderlineSMOTE, SMOTE
except ImportError:
    SMOTETomek = ADASYN = BorderlineSMOTE = SMOTE = None

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

try:
    import scikit_posthocs as sp
except ImportError:
    sp = None

# Constants
RANDOM_STATE = 42
METHODS = ("Baseline", "SMOTE", "B-SMOTE", "ADASYN", "SMOTE-Tomek", "N-LDF")
CLASSIFIERS = ("RF", "SVM-Linear", "SVM-RBF", "XGBoost")
GLOBAL_TIMEOUT = 3600

# FIXED DATASETS: Remove problematic URLs, use alternative sources
DATASETS: dict[str, str] = {
    # Mild imbalance
    "glass1": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/glass1.zip",
    "pima": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/pima.zip",
    "iris0": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/iris0.zip",
    "yeast1": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast1.zip",
    "haberman": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/haberman.zip",
    "vehicle1": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/vehicle1.zip",
    "ecoli1": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/ecoli1.zip",
    "ecoli2": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/ecoli2.zip",
    "segment0": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/segment0.zip",
    "yeast3": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast3.zip",

    # High imbalance
    "yeast-2_vs_4": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast-2_vs_4.zip",
    "vowel0": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/vowel0.zip",
    "glass4": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/glass4.zip",
    "ecoli4": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/ecoli4.zip",
    "abalone9-18": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/abalone9-18.zip",
    "yeast-1-4-5-8_vs_7": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast-1-4-5-8_vs_7.zip",
    "glass5": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/glass5.zip",
    "yeast-2_vs_8": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast-2_vs_8.zip",
    "yeast4": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast4.zip",
    "yeast-1-2-8-9_vs_7": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast-1-2-8-9_vs_7.zip",
    "yeast5": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast5.zip",
    "ecoli-0-1-3-7_vs_2-6": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/ecoli-0-1-3-7_vs_2-6.zip",

    # Extreme imbalance
    "abalone-21_vs_8": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/abalone-21_vs_8.zip",
    "yeast6": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast6.zip",
    "winequality-white-3_vs_7": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/winequality-white-3_vs_7.zip",
    "winequality-red-8_vs_6-7": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/winequality-red-8_vs_6-7.zip",
    "abalone-19_vs_10-11-12-13": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/abalone-19_vs_10-11-12-13.zip",
    "winequality-white-3-9_vs_5": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/winequality-white-3-9_vs_5.zip",
    "poker-8-9_vs_6": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/poker-8-9_vs_6.zip",
    "shuttle-2_vs_5": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/shuttle-2_vs_5.zip",
    "winequality-red-3_vs_5": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/winequality-red-3_vs_5.zip",
    "abalone-20_vs_8-9-10": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/abalone-20_vs_8-9-10.zip",
    "poker-8-9_vs_5": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/poker-8-9_vs_5.zip",
    "poker-8_vs_6": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/poker-8_vs_6.zip",
    "abalone19": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/abalone19.zip",

    # FIXED: KDD Cup datasets dengan alternative processing flag
    "kddcup-land_vs_portsweep": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/kddcup-land_vs_portsweep.zip",
    "kddcup-buffer_overflow_vs_back": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/kddcup-buffer_overflow_vs_back.zip",
    "kddcup-land_vs_satan": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/kddcup-land_vs_satan.zip",
    "kddcup-rootkit-imap_vs_back": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/kddcup-rootkit-imap_vs_back.zip",
    "kr-vs-k-zero_vs_eight": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/kr-vs-k-zero_vs_eight.zip",
    "kr-vs-k-zero_vs_fifteen": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/kr-vs-k-zero_vs_fifteen.zip",
}

# Flag problematic datasets yang butuh special handling
PROBLEMATIC_DATASETS = {
    "kddcup-land_vs_portsweep", "kddcup-buffer_overflow_vs_back",
    "kddcup-land_vs_satan", "kddcup-rootkit-imap_vs_back",
    "kr-vs-k-zero_vs_eight", "kr-vs-k-zero_vs_fifteen"
}

print(f"✅ Cell 1 loaded: {len(DATASETS)} datasets configured")
print(f"⚠️  Problematic datasets (special handling): {len(PROBLEMATIC_DATASETS)}")

### Cell 2: dataset loading, fold-wise preprocessing, classifiers, audited ADASYN

In [ ]:
"""Cell 2 (REVISED): Classes dengan Fixed Categorical Handling"""

@dataclass
class LoadedDataset:
    """Container for loaded dataset with metadata."""
    name: str
    X: pd.DataFrame
    y: np.ndarray
    numeric_columns: list[int]
    categorical_columns: list[int]
    n_samples: int
    n_features: int
    n_minority: int
    n_majority: int
    imbalance_ratio: float
    severity: str
    source_url: str
    sha256: str
    is_problematic: bool = False  # NEW: Flag for special handling


@dataclass
class CheckpointState:
    """State for checkpoint/resume functionality."""
    completed_datasets: set = field(default_factory=set)
    results: list[dict[str, object]] = field(default_factory=list)

    def save(self, path: Path) -> None:
        with open(path, 'wb') as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path: Path) -> 'CheckpointState':
        with open(path, 'rb') as f:
            return pickle.load(f)


class TimeoutError(Exception):
    pass


class FoldPreprocessor:
    """FOLD-PREPROCESSOR DENGAN FIX UNTUK KDD CUP DATASETS"""

    def __init__(self, numeric_columns: list[int], categorical_columns: list[int],
                 dataset_name: str = ""):
        self.numeric_columns = numeric_columns
        self.categorical_columns = categorical_columns
        self.dataset_name = dataset_name
        self.transformer: ColumnTransformer | None = None
        self.categorical_slices: list[slice] = []
        self.original_feature_order: list[int] = []
        self.has_categorical = len(categorical_columns) > 0
        self._fitted_categories: list = []

    def fit(self, X_train: pd.DataFrame) -> "FoldPreprocessor":
        """Fit preprocessor dengan safe categorical handling."""
        transformers: list[tuple[str, Pipeline, list[int]]] = []
        self.original_feature_order = []

        # Numeric pipeline
        if self.numeric_columns:
            numeric_pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ])
            transformers.append(("numeric", numeric_pipeline, self.numeric_columns))
            self.original_feature_order.extend(self.numeric_columns)

        # Categorical pipeline dengan FIX
        if self.categorical_columns:
            # Hanya proses kolom kategorikal yang valid
            valid_cat_cols = []
            for col in self.categorical_columns:
                if col < X_train.shape[1]:
                    n_unique = X_train.iloc[:, col].nunique()
                    # Skip jika terlalu banyak kategori (KDD Cup issue)
                    if n_unique < min(100, len(X_train) * 0.5):
                        valid_cat_cols.append(col)
                    else:
                        # Treat as numeric instead
                        if col not in self.numeric_columns:
                            self.numeric_columns.append(col)

            if valid_cat_cols:
                try:
                    encoder = OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                        dtype=float,
                        max_categories=50  # LIMIT categories untuk KDD Cup
                    )
                except TypeError:
                    encoder = OneHotEncoder(
                        handle_unknown="ignore",
                        sparse=False,
                        dtype=float
                    )

                categorical_pipeline = Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", encoder),
                ])
                transformers.append(("categorical", categorical_pipeline, valid_cat_cols))
                self.original_feature_order.extend(valid_cat_cols)
                self.categorical_columns = valid_cat_cols  # Update dengan yang valid saja

        if not transformers:
            # Fallback: semua kolom sebagai numerik
            all_cols = list(range(X_train.shape[1]))
            numeric_pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ])
            transformers.append(("numeric", numeric_pipeline, all_cols))
            self.numeric_columns = all_cols
            self.categorical_columns = []
            self.has_categorical = False

        self.transformer = ColumnTransformer(transformers, remainder="drop", sparse_threshold=0)
        self.transformer.fit(X_train)

        # Build categorical slices SAFELY
        self.categorical_slices = []
        if self.has_categorical and self.categorical_columns:
            try:
                if hasattr(self.transformer, 'transformers_'):
                    feature_idx = 0
                    for name, _, cols in self.transformer.transformers_:
                        if name == "numeric":
                            feature_idx += len(cols)
                        elif name == "categorical":
                            cat_transformer = self.transformer.named_transformers_["categorical"]
                            encoder = cat_transformer.named_steps["onehot"]
                            if hasattr(encoder, 'categories_'):
                                for categories in encoder.categories_:
                                    n_cats = len(categories)
                                    if n_cats > 0:  # Skip empty categories
                                        self.categorical_slices.append(
                                            slice(feature_idx, feature_idx + n_cats)
                                        )
                                        feature_idx += n_cats
            except Exception as e:
                # Jika gagal, treat all as numeric
                self.has_categorical = False
                self.categorical_slices = []

        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        """Transform data safely."""
        if self.transformer is None:
            raise RuntimeError("FoldPreprocessor must be fitted before transform")
        try:
            return np.asarray(self.transformer.transform(X), dtype=float)
        except Exception as e:
            # Fallback: hanya gunakan kolom numerik
            if self.numeric_columns:
                return np.asarray(X.iloc[:, self.numeric_columns], dtype=float)
            else:
                return np.asarray(X, dtype=float)

    def project_categorical(self, X: np.ndarray) -> np.ndarray:
        """FIXED: Project categorical dengan empty array handling."""
        if not self.has_categorical or not self.categorical_slices:
            return np.asarray(X, dtype=float)

        projected = np.asarray(X, dtype=float).copy()

        for block in self.categorical_slices:
            # SAFETY CHECKS
            if block.start >= projected.shape[1]:
                continue
            if block.stop > projected.shape[1]:
                continue

            values = projected[:, block]
            if values.size == 0 or values.shape[1] == 0:
                continue

            try:
                winners = np.argmax(values, axis=1)
                values[:] = 0.0
                values[np.arange(len(values)), winners] = 1.0
                projected[:, block] = values
            except (ValueError, IndexError):
                # Jika gagal, biarkan values apa adanya
                continue

        return projected


# Copy semua fungsi utility yang sama (timeout_handler, stable_seed, dll)
def timeout_handler(signum, frame):
    raise TimeoutError("Dataset processing timed out")

def stable_seed(*parts: object, base: int = RANDOM_STATE) -> int:
    payload = "|".join(map(str, (base, *parts))).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "little")

def severity_from_ir(ir: float) -> str:
    if ir < 9:
        return "Mild"
    if ir < 40:
        return "High"
    return "Extreme"

def make_http_session() -> requests.Session:
    retry = Retry(total=3, connect=3, read=3, backoff_factor=1.0,
                  status_forcelist=(429, 500, 502, 503, 504),
                  allowed_methods=("GET",))
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.headers.update({"User-Agent": "NLDF-reproducibility-study/1.0"})
    return session

def parse_attribute_type(line: str) -> bool:
    lowered = line.lower()
    return bool(re.search(r"\b(real|integer|numeric)\b", lowered))

def effective_k(y_train: np.ndarray, requested: int = 5) -> int:
    n_minority = int((y_train == 1).sum())
    if n_minority < 2:
        raise ValueError("At least two training minority observations are required")
    return min(requested, n_minority - 1)

def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray) -> dict[str, float]:
    sensitivity = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    specificity = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    return {
        "F1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Recall": sensitivity,
        "Specificity": specificity,
        "G-Mean": float(np.sqrt(sensitivity * specificity)),
        "AUC-ROC": roc_auc_score(y_true, y_score),
        "AUPRC": average_precision_score(y_true, y_score),
    }

def build_classifiers(seed: int) -> dict[str, object]:
    if XGBClassifier is None:
        raise RuntimeError("xgboost is required")

    classifiers = {
        "RF": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed),
        "SVM-Linear": SVC(C=1.0, kernel="linear", probability=True, random_state=seed),
        "SVM-RBF": SVC(C=1.0, kernel="rbf", gamma="scale", probability=True, random_state=seed),
        "XGBoost": XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                  objective="binary:logistic", eval_metric="logloss",
                                  n_jobs=-1, random_state=seed),
    }

    # Warm-up XGBoost
    X_warm = np.random.randn(10, 4)
    y_warm = np.random.randint(0, 2, 10)
    classifiers["XGBoost"].fit(X_warm, y_warm)

    return classifiers


# UPDATED: load_keel_dataset dengan problem flag
def load_keel_dataset(name: str, url: str, session: requests.Session,
                      cache_dir: Path) -> LoadedDataset:
    """Load dataset dengan problem flag."""
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"{name}.zip"
    payload: bytes | None = None

    if cache_path.exists():
        cached_payload = cache_path.read_bytes()
        if zipfile.is_zipfile(io.BytesIO(cached_payload)):
            payload = cached_payload

    if payload is None:
        response = session.get(url, timeout=60)
        response.raise_for_status()
        payload = response.content
        if not zipfile.is_zipfile(io.BytesIO(payload)):
            raise ValueError(f"{name}: URL did not return a valid ZIP archive")
        cache_path.write_bytes(payload)

    digest = hashlib.sha256(payload).hexdigest()
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        dat_files = [m for m in archive.namelist() if m.lower().endswith(".dat")]
        if not dat_files:
            raise ValueError(f"{name}: ZIP archive contains no .dat file")
        with archive.open(dat_files[0]) as handle:
            content = None
            for encoding in ['utf-8', 'latin-1', 'cp1252']:
                try:
                    content = handle.read().decode(encoding)
                    break
                except UnicodeDecodeError:
                    continue
            if content is None:
                raise ValueError(f"{name}: Unable to decode file")

    lines = content.splitlines()
    try:
        data_index = next(i for i, line in enumerate(lines)
                        if line.strip().lower() == "@data")
    except StopIteration as exc:
        raise ValueError(f"{name}: @data marker not found") from exc

    attribute_lines = [line.strip() for line in lines[:data_index]
                      if line.strip().lower().startswith("@attribute")]

    raw_rows = [line.strip() for line in lines[data_index + 1:]
                if line.strip() and not line.lstrip().startswith(("%", "@"))]

    frame = pd.read_csv(io.StringIO("\n".join(raw_rows)), header=None,
                       sep=r",\s*", engine="python", dtype=str,
                       keep_default_na=False).replace(
        {"?": np.nan, "<null>": np.nan, "NULL": np.nan, "": np.nan})

    feature_types = [parse_attribute_type(line) for line in attribute_lines[:-1]]
    numeric_columns = [i for i, is_num in enumerate(feature_types) if is_num]
    categorical_columns = [i for i, is_num in enumerate(feature_types) if not is_num]

    X = frame.iloc[:, :-1].copy()
    for column in numeric_columns:
        X[column] = pd.to_numeric(X[column], errors="coerce")
    for column in categorical_columns:
        X[column] = X[column].where(X[column].notna(), np.nan).astype(object)

    target = frame.iloc[:, -1]
    valid_target = target.notna()
    X = X.loc[valid_target].reset_index(drop=True)
    target = target.loc[valid_target].astype(str).reset_index(drop=True)

    labels, counts = np.unique(target, return_counts=True)
    if len(labels) != 2:
        raise ValueError(f"{name}: expected binary labels, found {len(labels)}")

    minority_label = labels[np.argmin(counts)]
    y = (target.to_numpy() == minority_label).astype(int)
    n_minority = int(y.sum())
    n_majority = int(len(y) - n_minority)

    ir = n_majority / n_minority if n_minority > 0 else float('inf')

    is_problematic = name in PROBLEMATIC_DATASETS

    return LoadedDataset(
        name=name, X=X, y=y,
        numeric_columns=numeric_columns,
        categorical_columns=categorical_columns,
        n_samples=len(y), n_features=X.shape[1],
        n_minority=n_minority, n_majority=n_majority,
        imbalance_ratio=ir,
        severity=severity_from_ir(ir),
        source_url=url, sha256=digest,
        is_problematic=is_problematic
    )


class AuditedADASYN:
    """ADASYN with SMOTE fallback."""
    def __init__(self, n_neighbors: int, random_state: int):
        self.n_neighbors = n_neighbors
        self.random_state = random_state
        self.last_diagnostics_: dict[str, object] = {}

    def fit_resample(self, X: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        self.last_diagnostics_ = {"SamplerStatus": "success",
                                   "SamplerFallback": "none", "SamplerError": ""}
        try:
            return ADASYN(n_neighbors=self.n_neighbors,
                         random_state=self.random_state).fit_resample(X, y)
        except (RuntimeError, ValueError) as exc:
            message = str(exc)
            if ("not suited" in message or "No samples" in message):
                X_res, y_res = SMOTE(k_neighbors=self.n_neighbors,
                                    random_state=self.random_state).fit_resample(X, y)
                self.last_diagnostics_ = {"SamplerStatus": "fallback",
                                         "SamplerFallback": "SMOTE",
                                         "SamplerError": message}
                return X_res, y_res
            raise

print("✅ Cell 2 loaded: Fixed FoldPreprocessor for KDD Cup datasets")

### Cell 3: N-LDF from `nldf.py` (hash-checked) and the sampler builder

In [ ]:
"""Cell 3: the N-LDF sampler, imported from nldf.py (not re-defined here).

nldf.py is the class that produced every archived result -- "Cell 3 (FINAL
FIXED)" of the executed notebook, transcribed verbatim -- together with
build_sampler, which fixes the sixteen benchmark parameter values. The class
defaults themselves are NOT the benchmark values (tau_boundary and
acceptance_threshold in particular); only build_sampler is authoritative.
The SHA-256 below is the one quoted in Section 5.4 of the manuscript.
"""
import hashlib

import nldf
from nldf import NLDFv2
from nldf import build_sampler as build_nldf_sampler
from nldf import effective_k as nldf_effective_k

NLDF_SHA256 = "77c5f72721508e0cf48ac92af162e53c784dabb516002e460f49b6d0fa2a9872"
_digest = hashlib.sha256(Path(nldf.__file__).read_bytes()).hexdigest()
if _digest != NLDF_SHA256:
    raise RuntimeError(f"nldf.py hash {_digest} differs from the archived implementation {NLDF_SHA256}")
print(f"nldf.py verified: SHA-256 {_digest[:8]}...{_digest[-4:]}")

# effective_k is defined identically in Cell 2 and in nldf.py; make sure of it.
_y = np.array([1] * 3 + [0] * 10)
assert effective_k(_y) == nldf_effective_k(_y) == 2


def build_sampler(method: str, y_train: np.ndarray,
                  projector: Callable[[np.ndarray], np.ndarray], seed: int):
    """Sampler used in the benchmark for `method`.

    Comparison oversamplers: imbalanced-learn defaults with
    k_neighbors = effective_k(y_train) (5, or n_min - 1 when smaller), exactly
    as in the executed notebook. N-LDF: nldf.build_sampler, which fixes every
    parameter to the archived value and disables the inner-CV option.
    """
    if method != "Baseline" and SMOTE is None:
        raise RuntimeError("imbalanced-learn required")
    k = effective_k(y_train)
    if method == "Baseline":
        return None
    if method == "SMOTE":
        return SMOTE(k_neighbors=k, random_state=seed)
    if method == "B-SMOTE":
        return BorderlineSMOTE(k_neighbors=k, random_state=seed)
    if method == "ADASYN":
        return AuditedADASYN(n_neighbors=k, random_state=seed)
    if method == "SMOTE-Tomek":
        return SMOTETomek(
            smote=SMOTE(k_neighbors=k, random_state=seed),
            random_state=seed,
        )
    if method == "N-LDF":
        return build_nldf_sampler("N-LDF", y_train, projector, seed)
    raise KeyError(f"Unknown method: {method}")


# The benchmark parameter values, read from the sampler object itself so that
# the environment report (Cell 9) cannot drift from the code.
_probe = build_nldf_sampler("N-LDF", np.array([1] * 30 + [0] * 170), lambda X: X, 42)
NLDF_BENCHMARK_PARAMETERS = {
    key: value for key, value in vars(_probe).items()
    if key not in ("projector", "last_diagnostics_", "random_state", "k_neighbors")
}
NLDF_BENCHMARK_PARAMETERS["k_neighbors"] = "min(5, n_min - 1), per training fold"
print("\nBenchmark parameter values (from nldf.build_sampler):")
for key, value in NLDF_BENCHMARK_PARAMETERS.items():
    print(f"   {key:22s} = {value}")

# Quick verification on synthetic data. The FINAL FIXED class is the one that
# writes the diagnostic keys n_attempted / n_rejected_* carried by the archive.
_rng = np.random.default_rng(42)
_X = _rng.normal(0, 1, (200, 5))
_y = np.zeros(200, dtype=int)
_y[:30] = 1
_sampler = build_sampler("N-LDF", _y, lambda X: X, 42)
assert isinstance(_sampler, NLDFv2)
_Xr, _yr = _sampler.fit_resample(_X, _y)
_required = {"n_attempted", "n_rejected_distance", "n_rejected_density",
             "n_rejected_maj", "acceptance_rate", "achieved_ratio"}
missing = _required - set(_sampler.last_diagnostics_)
assert not missing, f"diagnostic keys missing: {missing}"
print(f"\nverification: {_X.shape} -> {_Xr.shape}; minority {int(_y.sum())} -> {int(_yr.sum())}")
print("diagnostics  :", {k: _sampler.last_diagnostics_[k] for k in sorted(_required)})


### Cell 4: benchmark loop

In [ ]:
"""CELL 4 (FIXED): Benchmark dengan N-LDF Validation"""

def run_benchmark(
    datasets: dict[str, LoadedDataset],
    n_splits: int,
    n_repeats: int,
    checkpoint_path: Optional[Path] = None,
    failed_datasets: Optional[set[str]] = None,
    skip_failures: bool = False,  # Default False for debugging
) -> tuple[pd.DataFrame, set[str]]:
    """Run benchmark dengan validasi ketat untuk N-LDF."""

    checkpoint = CheckpointState()
    if checkpoint_path and checkpoint_path.exists():
        checkpoint = CheckpointState.load(checkpoint_path)
        print(f"Resuming from checkpoint: {len(checkpoint.completed_datasets)} processed")

    new_failures = set()
    expected_per_dataset = n_splits * n_repeats * len(METHODS) * len(CLASSIFIERS)

    for dataset_name, dataset in tqdm(datasets.items(), desc="Datasets"):
        if dataset_name in checkpoint.completed_datasets:
            continue

        if failed_datasets and dataset_name in failed_datasets:
            continue

        # Adaptive folds
        if dataset.is_problematic:
            actual_folds = min(n_splits, max(2, dataset.n_minority // 5))
            actual_repeats = min(n_repeats, 2)
        elif dataset.n_minority < n_splits * 2:
            actual_folds = max(2, dataset.n_minority // 2)
            actual_repeats = n_repeats
        else:
            actual_folds = n_splits
            actual_repeats = n_repeats

        print(f"\n📊 {dataset_name}: folds={actual_folds}, repeats={actual_repeats}, "
              f"minority={dataset.n_minority}")

        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(GLOBAL_TIMEOUT)

        dataset_results_count = 0
        nldf_count = 0
        dataset_failed = False

        try:
            splitter = RepeatedStratifiedKFold(
                n_splits=actual_folds,
                n_repeats=actual_repeats,
                random_state=stable_seed(dataset_name),
            )

            for split_index, (train_idx, test_idx) in enumerate(
                splitter.split(dataset.X, dataset.y)
            ):
                repeat = split_index // actual_folds
                fold = split_index % actual_folds

                X_train_raw = dataset.X.iloc[train_idx]
                X_test_raw = dataset.X.iloc[test_idx]
                y_train = dataset.y[train_idx]
                y_test = dataset.y[test_idx]

                if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
                    continue

                preprocessor = FoldPreprocessor(
                    dataset.numeric_columns,
                    dataset.categorical_columns,
                    dataset_name=dataset_name
                ).fit(X_train_raw)

                X_train = preprocessor.transform(X_train_raw)
                X_test = preprocessor.transform(X_test_raw)

                if np.isnan(X_train).any() or np.isnan(X_test).any():
                    continue

                for method in METHODS:
                    seed = stable_seed(dataset_name, repeat, fold, method)

                    try:
                        sampler = build_sampler(
                            method, y_train, preprocessor.project_categorical, seed
                        )

                        if sampler is None:
                            X_res, y_res = X_train.copy(), y_train.copy()
                            diag = {}
                        else:
                            X_res, y_res = sampler.fit_resample(X_train, y_train)
                            try:
                                X_res = preprocessor.project_categorical(X_res)
                            except Exception:
                                pass
                            diag = getattr(sampler, "last_diagnostics_", {})

                        if len(np.unique(y_res)) < 2:
                            continue

                        for clf_name in CLASSIFIERS:
                            try:
                                clf = build_classifiers(
                                    stable_seed(dataset_name, repeat, fold, clf_name)
                                )[clf_name]

                                clf.fit(X_res, y_res)
                                y_pred = clf.predict(X_test)
                                y_score = clf.predict_proba(X_test)[:, 1]
                                metrics = evaluate_predictions(y_test, y_pred, y_score)

                                checkpoint.results.append({
                                    "Dataset": dataset_name,
                                    "Severity": dataset.severity,
                                    "IR": dataset.imbalance_ratio,
                                    "Repeat": repeat,
                                    "Fold": fold,
                                    "Method": method,
                                    "Classifier": clf_name,
                                    "NTrain": len(train_idx),
                                    "NTest": len(test_idx),
                                    "NResampled": len(y_res),
                                    **metrics,
                                    **diag,
                                })
                                dataset_results_count += 1
                                if method == "N-LDF":
                                    nldf_count += 1

                            except Exception as e:
                                if method == "N-LDF" and not skip_failures:
                                    print(f"\n❌ N-LDF + {clf_name} FAILED:")
                                    traceback.print_exc()
                                    raise  # Don't hide N-LDF errors!
                                continue

                    except Exception as e:
                        if method == "N-LDF" and not skip_failures:
                            print(f"\n❌ N-LDF build/fit FAILED:")
                            traceback.print_exc()
                            raise  # Don't hide N-LDF errors!
                        continue

            # ============================================
            # VALIDASI SETELAH DATASET SELESAI
            # ============================================
            print(f"   Results: {dataset_results_count} rows, N-LDF: {nldf_count} rows")

            if dataset_results_count > 0:
                # Check N-LDF presence
                if nldf_count == 0:
                    msg = f"N-LDF produced 0 results for {dataset_name}!"
                    print(f"   ❌ {msg}")
                    if not skip_failures:
                        raise RuntimeError(msg)
                    else:
                        new_failures.add(dataset_name)
                        continue

                checkpoint.completed_datasets.add(dataset_name)
                if checkpoint_path:
                    checkpoint.save(checkpoint_path)
            else:
                print(f"   ❌ No results at all!")
                if not skip_failures:
                    raise RuntimeError(f"No results for {dataset_name}")
                new_failures.add(dataset_name)

        except TimeoutError:
            print(f"   ⏰ Timeout")
            new_failures.add(dataset_name)
        except Exception as e:
            print(f"   ❌ Error: {e}")
            if not skip_failures:
                raise
            new_failures.add(dataset_name)
        finally:
            signal.alarm(0)

    return pd.DataFrame(checkpoint.results), new_failures


print("✅ Cell 4 FIXED: Benchmark dengan N-LDF validation")
print("   - Akan RAISE ERROR jika N-LDF gagal")
print("   - skip_failures=False by default")

### Cell 5: dataset diagnostics

In [ ]:
"""Cell 4.5: Dataset Diagnostics - Memeriksa Dataset Sebelum Benchmark"""

def diagnose_datasets(loaded_datasets: dict[str, LoadedDataset]):
    """Check datasets for potential issues before benchmark."""
    print("="*60)
    print("📊 DATASET DIAGNOSTICS")
    print("="*60)

    problematic = []
    healthy = []

    for name, ds in loaded_datasets.items():
        issues = []

        # Check minority samples
        if ds.n_minority < 10:
            issues.append(f"⚠️  Very few minority samples: {ds.n_minority}")

        # Check imbalance ratio
        if ds.imbalance_ratio > 100:
            issues.append(f"⚠️  Extreme imbalance: {ds.imbalance_ratio:.1f}:1")
        elif ds.imbalance_ratio > 50:
            issues.append(f"⚡ High imbalance: {ds.imbalance_ratio:.1f}:1")

        # Check dimensionality
        if ds.n_features > 100:
            issues.append(f"📐 High dimensional: {ds.n_features} features")

        # Check sample size
        if ds.n_samples < 50:
            issues.append(f"📏 Very small dataset: {ds.n_samples} samples")

        # Check categorical features
        if len(ds.categorical_columns) > ds.n_features * 0.5:
            issues.append(f"🔤 Many categorical features: {len(ds.categorical_columns)}/{ds.n_features}")

        if issues:
            problematic.append((name, ds, issues))
            print(f"\n❌ {name}:")
            print(f"   Samples: {ds.n_samples} | Features: {ds.n_features}")
            print(f"   Minority: {ds.n_minority} | IR: {ds.imbalance_ratio:.1f} | Severity: {ds.severity}")
            for issue in issues:
                print(f"   {issue}")
        else:
            healthy.append(name)
            print(f"✅ {name}: OK (n={ds.n_samples}, IR={ds.imbalance_ratio:.1f}, min={ds.n_minority})")

    print(f"\n{'='*60}")
    print(f"📈 DIAGNOSTICS SUMMARY")
    print(f"{'='*60}")
    print(f"   ✅ Healthy datasets: {len(healthy)}/{len(loaded_datasets)}")
    print(f"   ⚠️  Potentially problematic: {len(problematic)}/{len(loaded_datasets)}")

    if problematic:
        print(f"\n   Problematic datasets:")
        for name, ds, issues in problematic:
            print(f"   - {name}: {len(issues)} issue(s) | IR={ds.imbalance_ratio:.1f} | Minority={ds.n_minority}")

    # Severity distribution
    severity_count = {}
    for ds in loaded_datasets.values():
        severity_count[ds.severity] = severity_count.get(ds.severity, 0) + 1

    print(f"\n   Severity distribution:")
    for severity in ["Mild", "High", "Extreme"]:
        count = severity_count.get(severity, 0)
        print(f"   - {severity}: {count} datasets")

    return problematic, healthy


def suggest_cv_params(datasets: dict[str, LoadedDataset], default_folds: int = 5):
    """Suggest CV parameters based on dataset characteristics."""
    print(f"\n{'='*60}")
    print(f"🔧 CV PARAMETER SUGGESTIONS")
    print(f"{'='*60}")

    min_minority = min(ds.n_minority for ds in datasets.values())
    max_minority = max(ds.n_minority for ds in datasets.values())
    avg_minority = np.mean([ds.n_minority for ds in datasets.values()])

    print(f"   Minority samples - Min: {min_minority}, Max: {max_minority}, Avg: {avg_minority:.0f}")

    if min_minority < default_folds * 2:
        suggested_folds = max(2, min_minority // 2)
        print(f"   ⚠️  Minimum minority samples ({min_minority}) too low for {default_folds}-fold CV")
        print(f"   💡 Suggestion: Use {suggested_folds}-fold CV for small datasets")
    else:
        print(f"   ✅ Minimum minority samples ({min_minority}) sufficient for {default_folds}-fold CV")

    # List datasets that need reduced folds
    needs_reduction = []
    for name, ds in datasets.items():
        if ds.n_minority < default_folds * 2:
            needs_reduction.append((name, ds.n_minority, max(2, ds.n_minority // 2)))

    if needs_reduction:
        print(f"\n   Datasets needing fold reduction:")
        for name, min_count, suggested in sorted(needs_reduction, key=lambda x: x[1]):
            print(f"   - {name:30s}: {min_count} minority → suggest {suggested} folds")


# Diagnosa dataset jika sudah di-load
if 'loaded' in locals() and loaded:
    print("🔍 Running dataset diagnostics...\n")
    problems, healthy = diagnose_datasets(loaded)
    suggest_cv_params(loaded)
else:
    print("ℹ️  No datasets loaded yet.")
    print("   Datasets will be automatically diagnosed during benchmark.")
    print("   Run Cell 6 to load and diagnose datasets.")

print(f"\n✅ Cell 4.5 completed: Diagnostics ready")

### Cell 5b: statistical analysis helpers used by the loop

In [ ]:
"""Cell 5: Statistical Analysis Functions"""

def bootstrap_confidence_interval(
    values: np.ndarray,
    confidence: float = 0.95,
    n_bootstrap: int = 10000,
) -> tuple[float, float, float, float]:
    """Calculate bootstrap confidence interval with effect size."""
    data = values[~np.isnan(values)]
    if len(data) < 2:
        return float("nan"), float("nan"), float("nan"), float("nan")

    rng = np.random.default_rng(RANDOM_STATE)
    bootstrap_means = []
    for _ in range(n_bootstrap):
        sample = rng.choice(data, size=len(data), replace=True)
        bootstrap_means.append(np.mean(sample))

    bootstrap_means = np.array(bootstrap_means)
    lower = np.percentile(bootstrap_means, ((1 - confidence) / 2) * 100)
    upper = np.percentile(bootstrap_means, (1 - (1 - confidence) / 2) * 100)

    # Calculate Cohen's d effect size
    cohens_d = np.mean(data) / np.std(data) if np.std(data) > 0 else 0.0

    # Calculate Cliff's delta for non-parametric effect size
    cliff_delta = (2 * np.mean(data > np.median(data))) - 1 if len(data) > 1 else 0.0

    return lower, upper, cohens_d, cliff_delta


def holm_adjust(p_values: list[float]) -> list[float]:
    """Apply Holm-Bonferroni correction for multiple comparisons."""
    order = np.argsort(p_values)
    adjusted = np.empty(len(p_values), dtype=float)
    running = 0.0
    total = len(p_values)
    for rank, index in enumerate(order):
        candidate = min(1.0, (total - rank) * p_values[index])
        running = max(running, candidate)
        adjusted[index] = running
    return adjusted.tolist()


def statistical_analysis(results: pd.DataFrame, output_dir: Path) -> None:
    """Perform comprehensive statistical analysis."""
    if sp is None:
        raise RuntimeError(
            "scikit-posthocs is required for the Nemenyi analysis. "
            "Install the declared project requirements before the full run."
        )

    # Aggregate results
    dataset_method = (
        results.groupby(["Dataset", "Method"], as_index=False)[
            ["F1", "Precision", "Recall", "G-Mean", "AUC-ROC", "AUPRC"]
        ]
        .mean()
    )

    pivot = dataset_method.pivot(index="Dataset", columns="Method", values="F1")
    pivot = pivot.reindex(columns=METHODS)

    # Validate pivot table
    if pivot.shape != (len(results["Dataset"].unique()), len(METHODS)) or pivot.isna().any().any():
        print(f"Warning: Statistical matrix incomplete. Shape: {pivot.shape}, NaN count: {int(pivot.isna().sum().sum())}")
        # Fill missing values with worst possible value instead of failing
        pivot = pivot.fillna(0.0)

    # Friedman test
    ranks = pivot.rank(axis=1, ascending=False, method="average")
    friedman_statistic, friedman_p = stats.friedmanchisquare(
        *[pivot[column].to_numpy() for column in METHODS if column in pivot.columns]
    )

    # Nemenyi post-hoc test
    metody_available = [m for m in METHODS if m in pivot.columns]
    if len(metody_available) == len(METHODS):
        pivot_for_nemenyi = pivot[METHODS]
    else:
        pivot_for_nemenyi = pivot[metody_available]

    nemenyi = sp.posthoc_nemenyi_friedman(pivot_for_nemenyi.to_numpy())
    nemenyi.index = list(pivot_for_nemenyi.columns)
    nemenyi.columns = list(pivot_for_nemenyi.columns)
    nemenyi.to_csv(output_dir / "nemenyi_adjusted_p_values.csv")

    # Summary with bootstrap confidence intervals and effect sizes
    summary_rows: list[dict[str, object]] = []
    for method in METHODS:
        if method not in dataset_method["Method"].values:
            continue

        method_values = dataset_method.loc[dataset_method["Method"] == method]
        f1_values = method_values["F1"].to_numpy()
        f1_low, f1_high, cohens_d, cliff_delta = bootstrap_confidence_interval(f1_values)

        summary_rows.append({
            "Method": method,
            "N_Datasets": len(method_values),
            "F1": method_values["F1"].mean(),
            "F1_CI95_Low": f1_low,
            "F1_CI95_High": f1_high,
            "Cohens_d": cohens_d,
            "Cliffs_delta": cliff_delta,
            "Precision": method_values["Precision"].mean(),
            "Recall": method_values["Recall"].mean(),
            "G-Mean": method_values["G-Mean"].mean(),
            "AUC-ROC": method_values["AUC-ROC"].mean(),
            "AUPRC": method_values["AUPRC"].mean(),
            "MeanRank": ranks[method].mean() if method in ranks.columns else float("nan"),
            "FriedmanStatistic": friedman_statistic,
            "FriedmanDF": len(METHODS) - 1,
            "FriedmanP": friedman_p,
        })

    pd.DataFrame(summary_rows).sort_values("MeanRank").to_csv(
        output_dir / "global_summary.csv", index=False
    )

    # Complete pairwise Wilcoxon tests with effect sizes
    if len(metody_available) >= 2:
        pairwise_results = []
        for method1 in metody_available:
            for method2 in metody_available:
                if method1 >= method2:  # Avoid duplicates
                    continue

                statistic, p_value = stats.wilcoxon(
                    pivot[method1].to_numpy(),
                    pivot[method2].to_numpy(),
                    zero_method="pratt",
                    alternative="two-sided",
                )

                # Calculate effect size (matched-pairs rank-biserial correlation)
                differences = pivot[method1].to_numpy() - pivot[method2].to_numpy()
                non_zero_diff = differences[differences != 0]
                if len(non_zero_diff) > 0:
                    rank_biserial = np.mean(np.sign(non_zero_diff))
                else:
                    rank_biserial = 0.0

                pairwise_results.append({
                    "Method1": method1,
                    "Method2": method2,
                    "Statistic": statistic,
                    "RawP": p_value,
                    "RankBiserialCorrelation": rank_biserial,
                })

        # Apply Holm correction to all pairwise p-values
        if pairwise_results:
            raw_p_values = [r["RawP"] for r in pairwise_results]
            adjusted_p = holm_adjust(raw_p_values)
            for result, adj_p in zip(pairwise_results, adjusted_p):
                result["HolmP"] = adj_p
                result["SignificantAt0.05"] = adj_p < 0.05

            pd.DataFrame(pairwise_results).to_csv(
                output_dir / "pairwise_wilcoxon.csv", index=False
            )

    # Severity analysis
    severity_summary = (
        results.groupby(["Severity", "Method"], as_index=False)[
            ["F1", "G-Mean", "AUPRC"]
        ]
        .mean()
    )
    severity_summary.to_csv(output_dir / "severity_summary.csv", index=False)

    # Dataset-method means
    dataset_method.to_csv(output_dir / "dataset_method_means.csv", index=False)

    # Sampler fallback events
    fallback_columns = [
        "Dataset", "Severity", "Repeat", "Fold", "Method",
        "SamplerStatus", "SamplerFallback", "SamplerError",
    ]
    if all(column in results.columns for column in fallback_columns):
        fallback_events = (
            results.loc[results["SamplerStatus"].notna(), fallback_columns]
            .drop_duplicates()
            .reset_index(drop=True)
        )
        fallback_events.to_csv(output_dir / "sampler_events.csv", index=False)
        fallback_summary = (
            fallback_events.groupby(
                ["Method", "SamplerStatus", "SamplerFallback"],
                as_index=False,
                dropna=False,
            )
            .size()
            .rename(columns={"size": "NFolds"})
        )
        fallback_summary.to_csv(output_dir / "sampler_fallback_summary.csv", index=False)


def write_environment(output_dir: Path, arguments: argparse.Namespace) -> None:
    """Write environment information for reproducibility."""
    packages = [
        "numpy", "pandas", "scipy", "scikit-learn",
        "imbalanced-learn", "xgboost", "scikit-posthocs", "requests",
    ]
    versions: dict[str, str] = {}
    for package in packages:
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = "not-installed"

    serialized_arguments = {
        key: str(value) if isinstance(value, Path) else value
        for key, value in vars(arguments).items()
    }
    payload = {
        "python": sys.version,
        "platform": platform.platform(),
        "arguments": serialized_arguments,
        "packages": versions,
    }
    (output_dir / "environment.json").write_text(
        json.dumps(payload, indent=2), encoding="utf-8"
    )


def parse_args() -> argparse.Namespace:
    """Parse command line arguments with notebook support."""
    parser = argparse.ArgumentParser(description="N-LDF Benchmark")
    parser.add_argument("--folds", type=int, default=5)
    parser.add_argument("--repeats", type=int, default=5)
    parser.add_argument("--output-dir", type=Path, default=Path("nldf_results_41"))
    parser.add_argument("--cache-dir", type=Path, default=Path("keel_cache"),
                       help="Separate cache directory for downloaded datasets")
    parser.add_argument("--skip-failures", action="store_true",
                       help="Continue with remaining datasets if some fail")
    parser.add_argument("--checkpoint", type=Path,
                       help="Path to checkpoint file for resume capability")
    parser.add_argument("--timeout", type=int, default=3600,
                       help="Timeout in seconds per dataset (default: 3600)")
    parser.add_argument("--quick-test", action="store_true",
                       help="Run on subset of datasets with fewer repeats for testing")

    running_in_notebook = "ipykernel" in sys.modules or "google.colab" in sys.modules
    if running_in_notebook:
        arguments, unknown = parser.parse_known_args()
        remaining = list(unknown)
        index = 0
        while index < len(remaining):
            if remaining[index] == "-f" and index + 1 < len(remaining):
                index += 2
                continue
            if remaining[index].startswith("--f="):
                index += 1
                continue
            parser.error(f"unrecognized notebook argument: {remaining[index]}")
        return arguments

    return parser.parse_args()

print("✅ Cell 5 completed: Analysis functions loaded")

### Cell 6: run (set `SMOKE_TEST` / `CLEAN_START` at the top of the cell)

In [ ]:
"""Cell 6: FULL BENCHMARK, 41 datasets, N-LDF from nldf.py.

Set SMOKE_TEST = True to run the three-dataset, two-repeat check that
preceded the archived run (glass1, winequality-white-3_vs_7,
kddcup-land_vs_portsweep). Set CLEAN_START = True to delete a previous
output directory and checkpoint before running.
"""

SMOKE_TEST = False
CLEAN_START = False
SMOKE_DATASETS = ["glass1", "winequality-white-3_vs_7", "kddcup-land_vs_portsweep"]

import shutil
from datetime import datetime

# ============================================================
# KONFIGURASI FULL BENCHMARK
# ============================================================
FULL_RUN_CONFIG = {
    "n_folds": 5,
    "n_repeats": 2 if SMOKE_TEST else 5,
    "output_dir": Path("smoke_test_output" if SMOKE_TEST else "nldf_results_41_v2_final"),
    "cache_dir": Path("keel_cache"),
    "checkpoint_path": Path("checkpoint_smoke.pkl" if SMOKE_TEST else "checkpoint_v2_final.pkl"),
    "skip_failures": True,      # True untuk full run (jangan berhenti)
    "timeout_per_dataset": 3600, # 1 jam
    "use_all_41_datasets": True,
}

print("="*80)
print("🚀 FULL BENCHMARK 41 DATASETS - N-LDF v2.0")
print("="*80)
print(f"Configuration:")
print(f"  Datasets: {'3 (smoke test)' if SMOKE_TEST else 'ALL 41'}")
print(f"  CV: {FULL_RUN_CONFIG['n_folds']}-fold × {FULL_RUN_CONFIG['n_repeats']} repeats")
print(f"  Methods: {METHODS}")
print(f"  Classifiers: {CLASSIFIERS}")
print(f"  Expected experiments: 41 × 25 × 6 × 4 = 24,600")
print(f"  skip_failures: {FULL_RUN_CONFIG['skip_failures']}")
print(f"  Timeout/dataset: {FULL_RUN_CONFIG['timeout_per_dataset']}s")
print(f"  Estimated total time: ~6-8 hours")

# ============================================================
# CLEANUP OPSIONAL
# ============================================================
if CLEAN_START:
    print("🧹 Cleaning...")
    for path in [
        FULL_RUN_CONFIG['output_dir'],
        FULL_RUN_CONFIG['checkpoint_path'],
        Path("nldf_results_41_fixed"),
        Path("nldf_results_41_fixed_v2"),
        Path("checkpoint_fixed.pkl"),
        Path("checkpoint_fixed_v2.pkl"),
    ]:
        if path.exists():
            if path.is_dir():
                shutil.rmtree(path)
            else:
                path.unlink()
            print(f"  Deleted: {path}")
    print("✅ Clean environment ready")

# ============================================================
# LOAD ALL 41 DATASETS
# ============================================================
print(f"\n📥 Phase 1: Loading datasets...")
print(f"   Cache: {FULL_RUN_CONFIG['cache_dir']}")

session = make_http_session()
loaded = {}
load_failures = []

DATASETS_TO_RUN = ({k: DATASETS[k] for k in SMOKE_DATASETS} if SMOKE_TEST else DATASETS)
for name, url in tqdm(DATASETS_TO_RUN.items(), desc="Loading datasets"):
    try:
        ds = load_keel_dataset(name, url, session, FULL_RUN_CONFIG['cache_dir'])
        loaded[name] = ds
    except Exception as e:
        load_failures.append({"Dataset": name, "Error": str(e)[:150]})
        print(f"  ❌ {name}: {str(e)[:80]}")

print(f"\n✅ Loaded: {len(loaded)}/{len(DATASETS_TO_RUN)} datasets")
print(f"❌ Failed to load: {len(load_failures)}")

if load_failures:
    pd.DataFrame(load_failures).to_csv(
        FULL_RUN_CONFIG['output_dir'] / "load_failures.csv", index=False
    )

# ============================================================
# DATASET SUMMARY
# ============================================================
print(f"\n📊 Dataset Summary:")
severity_count = {}
total_minority = 0
for ds in loaded.values():
    severity_count[ds.severity] = severity_count.get(ds.severity, 0) + 1
    total_minority += ds.n_minority

for sev in ['Mild', 'High', 'Extreme']:
    print(f"  {sev}: {severity_count.get(sev, 0)} datasets")
print(f"  Total minority samples: {total_minority}")
print(f"  Problematic datasets: {sum(1 for ds in loaded.values() if ds.is_problematic)}")

# ============================================================
# SAVE AUDIT
# ============================================================
FULL_RUN_CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)

audit_rows = []
for ds in loaded.values():
    audit_rows.append({
        "name": ds.name,
        "n_samples": ds.n_samples,
        "n_features": ds.n_features,
        "n_minority": ds.n_minority,
        "n_majority": ds.n_majority,
        "imbalance_ratio": round(ds.imbalance_ratio, 2),
        "severity": ds.severity,
        "is_problematic": ds.is_problematic,
        "n_numeric": len(ds.numeric_columns),
        "n_categorical": len(ds.categorical_columns),
        "sha256": ds.sha256,   # of the KEEL zip as downloaded (not recorded by the archived run)
    })

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(FULL_RUN_CONFIG['output_dir'] / "dataset_audit.csv", index=False)
print(f"\n💾 Audit saved")

# ============================================================
# RUN FULL BENCHMARK
# ============================================================
print(f"\n{'='*60}")
print(f"🚀 Phase 2: Running benchmark on {len(loaded)} datasets")
print(f"   Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Checkpoint: {FULL_RUN_CONFIG['checkpoint_path']}")
print(f"   You can interrupt and resume later")
print(f"{'='*60}\n")

GLOBAL_TIMEOUT = FULL_RUN_CONFIG['timeout_per_dataset']

start_time = datetime.now()

results, new_failures = run_benchmark(
    loaded,
    FULL_RUN_CONFIG['n_folds'],
    FULL_RUN_CONFIG['n_repeats'],
    checkpoint_path=FULL_RUN_CONFIG['checkpoint_path'],
    failed_datasets=set(f["Dataset"] for f in load_failures),
    skip_failures=FULL_RUN_CONFIG['skip_failures'],
)

elapsed = datetime.now() - start_time

# ============================================================
# SAVE RESULTS
# ============================================================
print(f"\n{'='*60}")
print(f"📊 Phase 3: Saving results")
print(f"{'='*60}")

results_path = FULL_RUN_CONFIG['output_dir'] / "fold_level_results.csv"
results.to_csv(results_path, index=False)
print(f"💾 Full results: {results_path}")

# ============================================================
# QUICK SUMMARY
# ============================================================
print(f"\n{'='*60}")
print(f"📈 QUICK SUMMARY")
print(f"{'='*60}")

print(f"Elapsed time: {elapsed}")
print(f"Total experiments: {len(results)}")

# Methods check
methods_found = sorted(results['Method'].unique()) if len(results) > 0 else []
print(f"Methods: {methods_found}")

# N-LDF check
nldf_rows = len(results[results['Method'] == 'N-LDF']) if len(results) > 0 else 0
print(f"N-LDF experiments: {nldf_rows}")

# Datasets with N-LDF
if nldf_rows > 0:
    nldf_datasets = sorted(results[results['Method'] == 'N-LDF']['Dataset'].unique())
    print(f"N-LDF datasets: {len(nldf_datasets)}/41")
else:
    print(f"❌ N-LDF STILL HAS 0 RESULTS!")

# Performance summary
if len(results) > 0:
    print(f"\nPerformance by method:")
    for method in METHODS:
        method_data = results[results['Method'] == method]
        if len(method_data) > 0:
            avg_f1 = method_data['F1'].mean()
            avg_gm = method_data['G-Mean'].mean()
            avg_auc = method_data['AUC-ROC'].mean()
            n_datasets = method_data['Dataset'].nunique()
            print(f"  {method:15s}: F1={avg_f1:.4f}, G-Mean={avg_gm:.4f}, "
                  f"AUC={avg_auc:.4f} ({n_datasets} datasets)")

# Failures
all_failures = set(f["Dataset"] for f in load_failures) | new_failures
print(f"\nFailed datasets: {len(all_failures)}")
if all_failures:
    print(f"  {sorted(all_failures)}")
    pd.DataFrame({"FailedDataset": sorted(list(all_failures))}).to_csv(
        FULL_RUN_CONFIG['output_dir'] / "failed_datasets.csv", index=False
    )

# ============================================================
# STATISTICAL ANALYSIS (if enough data)
# ============================================================
if len(results) > 1000 and nldf_rows > 0:
    print(f"\n{'='*60}")
    print(f"📈 Phase 4: Statistical analysis")
    print(f"{'='*60}")

    try:
        statistical_analysis(results, FULL_RUN_CONFIG['output_dir'])
        print("✅ Analysis complete")
    except Exception as e:
        print(f"⚠️  Analysis error: {e}")
        # Basic fallback
        summary = results.groupby('Method').agg({
            'F1': ['mean', 'std'],
            'G-Mean': ['mean', 'std'],
            'AUC-ROC': ['mean', 'std'],
            'AUPRC': ['mean', 'std'],
        }).round(4)
        summary.to_csv(FULL_RUN_CONFIG['output_dir'] / "basic_summary.csv")
        print(f"💾 Basic summary saved")

# ============================================================
# FINAL REPORT
# ============================================================
print(f"\n{'='*80}")
print(f"✅ FULL BENCHMARK COMPLETE")
print(f"{'='*80}")
print(f"📅 Start: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📅 End: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"⏱️  Duration: {elapsed}")
print(f"📊 Experiments: {len(results)}")
print(f"📁 Output: {FULL_RUN_CONFIG['output_dir'].absolute()}")
print(f"🔍 N-LDF rows: {nldf_rows}")

# Download instructions for Colab
print(f"\n💡 To download results from Colab:")
print(f"   from google.colab import files")
print(f"   !zip -r nldf_results_v2.zip {FULL_RUN_CONFIG['output_dir']}")
print(f"   files.download('nldf_results_v2.zip')")
print(f"{'='*80}")

### Cell 7: post-processing (repository script)

In [ ]:
"""Cell 7: post-processing with the repository script.

src/nldf_postprocessing.py is the code that produced results/dataset_method_means.csv
and the other archived summaries; it is run here unchanged on the fresh records.
"""
import subprocess
import sys

script = Path("src/nldf_postprocessing.py")
if not script.exists():
    script = Path("nldf_postprocessing.py")
run_dir = FULL_RUN_CONFIG["output_dir"]
subprocess.run([sys.executable, str(script),
                "--input", str(run_dir / "fold_level_results.csv"),
                "--audit", str(run_dir / "dataset_audit.csv"),
                "--output-dir", str(run_dir / "post_processing")], check=True)
print("post-processing written to", run_dir / "post_processing")


### Cell 8: checks against the manuscript (archived records)

In [ ]:
"""Cell 8: the manuscript's own checks (archived records only).

Both scripts recompute every printed number from results/fold_level_results.csv
and compare with the manuscript. They verify the ARCHIVED run; a fresh run
produces its own numbers and is not expected to pass them.
"""
import subprocess
import sys

if Path("results/fold_level_results.csv").exists():
    for script in ("reproduce_paper_tables.py", "supplementary_analysis.py"):
        print(f"\n$ python {script} --check")
        subprocess.run([sys.executable, script, "--check"], check=False)
else:
    print("results/fold_level_results.csv not found (not in the repository root?) - skipped")


### Cell 9: environment and reproducibility report

In [ ]:
"""Cell 9: COMPUTATIONAL ENVIRONMENT & REPRODUCIBILITY REPORT

Parameter values come from NLDF_BENCHMARK_PARAMETERS (Cell 3), i.e. from
nldf.build_sampler itself, not from a retyped dictionary.
"""

import sys
import platform
import importlib.metadata
import datetime
import os
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb

print("="*80)
print("🖥️  COMPUTATIONAL ENVIRONMENT & REPRODUCIBILITY REPORT")
print("="*80)

# ============================================================
# 1. SYSTEM INFORMATION
# ============================================================
print("\n" + "="*60)
print("1. SYSTEM INFORMATION")
print("="*60)

system_info = {
    "timestamp": datetime.datetime.now().isoformat(),
    "hostname": platform.node(),
    "os": platform.system(),
    "os_version": platform.version(),
    "os_release": platform.release(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "python_version": sys.version,
    "python_implementation": platform.python_implementation(),
    "python_compiler": platform.python_compiler(),
}

for key, value in system_info.items():
    print(f"   {key:25s}: {value}")

# ============================================================
# 2. HARDWARE INFORMATION (Google Colab Specific)
# ============================================================
print("\n" + "="*60)
print("2. HARDWARE INFORMATION")
print("="*60)

hardware_info = {}

# CPU info
try:
    import psutil
    hardware_info["cpu_count_logical"] = psutil.cpu_count(logical=True)
    hardware_info["cpu_count_physical"] = psutil.cpu_count(logical=False)
    hardware_info["cpu_freq_mhz"] = psutil.cpu_freq().max if psutil.cpu_freq() else "N/A"
    hardware_info["ram_total_gb"] = round(psutil.virtual_memory().total / (1024**3), 2)
    hardware_info["ram_available_gb"] = round(psutil.virtual_memory().available / (1024**3), 2)
except ImportError:
    hardware_info["psutil"] = "not installed"
    print("   ⚠️  psutil not installed (run: !pip install psutil)")

# GPU info
try:
    gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null
    if gpu_info and gpu_info[0]:
        gpu_name, gpu_mem = gpu_info[0].split(", ")
        hardware_info["gpu_name"] = gpu_name.strip()
        hardware_info["gpu_memory_mb"] = gpu_mem.strip()
    else:
        hardware_info["gpu"] = "Not available"
except Exception:
    hardware_info["gpu"] = "Not detected"

# Check Colab specifics
is_colab = "google.colab" in sys.modules
hardware_info["is_google_colab"] = is_colab
if is_colab:
    try:
        from google.colab import drive
        hardware_info["colab_runtime"] = "Connected"
    except:
        hardware_info["colab_runtime"] = "Not mounted"

for key, value in hardware_info.items():
    print(f"   {key:25s}: {value}")

# ============================================================
# 3. PACKAGE VERSIONS
# ============================================================
print("\n" + "="*60)
print("3. PACKAGE VERSIONS")
print("="*60)

packages = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": None,
    "scikit-learn": sklearn.__version__,
    "xgboost": xgb.__version__,
    "imbalanced-learn": None,
    "scikit-posthocs": None,
    "requests": None,
    "tqdm": None,
    "matplotlib": None,
    "seaborn": None,
}

# Get versions dynamically
for pkg_name in packages:
    if packages[pkg_name] is None:
        try:
            packages[pkg_name] = importlib.metadata.version(pkg_name)
        except importlib.metadata.PackageNotFoundError:
            packages[pkg_name] = "NOT INSTALLED"

for pkg, ver in packages.items():
    status = "✅" if ver != "NOT INSTALLED" else "❌"
    print(f"   {status} {pkg:25s}: {ver}")

# ============================================================
# 4. BENCHMARK CONFIGURATION
# ============================================================
print("\n" + "="*60)
print("4. BENCHMARK CONFIGURATION")
print("="*60)

benchmark_config = {
    "random_state": 42,
    "methods": ["Baseline", "SMOTE", "B-SMOTE", "ADASYN", "SMOTE-Tomek", "N-LDF"],
    "classifiers": ["RF", "SVM-Linear", "SVM-RBF", "XGBoost"],
    "cv_folds": 5,
    "cv_repeats": 5,
    "total_datasets": 41,
    "nldf_version": "v2.0",
    "nldf_parameters": NLDF_BENCHMARK_PARAMETERS,   # read from nldf.build_sampler in Cell 3
    "nldf_improvements": [
        "1. Class-prior-corrected density estimation",
        "2. Adaptive kernel with local bandwidth",
        "3. Quantile-based boundary (widens with IR)",
        "4. Soft seed weighting (continuous)",
        "5. Safe-directed neighbor selection",
        "6. Local covariance jitter",
        "7. Acceptance filtering",
        "8. Inner-CV sampling-ratio option present in the class, DISABLED in the benchmark (use_inner_cv=False; ratio fixed at 0.75)",
    ],
    "total_expected_experiments": 24600,
    "evaluation_metrics": ["F1", "G-Mean", "AUC-ROC", "AUPRC", "Precision", "Recall", "Specificity"],
}

for key, value in benchmark_config.items():
    if isinstance(value, list):
        print(f"   {key:25s}:")
        for item in value[:3]:
            print(f"      - {item}")
        if len(value) > 3:
            print(f"      ... and {len(value)-3} more")
    elif isinstance(value, dict):
        print(f"   {key:25s}:")
        for k, v in list(value.items())[:5]:
            print(f"      {k}: {v}")
        if len(value) > 5:
            print(f"      ... and {len(value)-5} more")
    else:
        print(f"   {key:25s}: {value}")

# ============================================================
# 5. DATASET INFORMATION
# ============================================================
print("\n" + "="*60)
print("5. DATASET INFORMATION")
print("="*60)

# Try to load dataset audit
audit_paths = [
    Path("nldf_results_41_v2_final/dataset_audit.csv"),
    Path("analysis_output/dataset_audit.csv"),
    Path("nldf_results_41_fixed/dataset_audit.csv"),
]

audit_found = False
for ap in audit_paths:
    if ap.exists():
        audit = pd.read_csv(ap)
        audit_found = True

        print(f"   Source: {ap}")
        print(f"   Total datasets: {len(audit)}")

        # Severity distribution
        if 'severity' in audit.columns:
            sev_dist = audit['severity'].value_counts().to_dict()
            print(f"   Severity distribution:")
            for sev in ['Mild', 'High', 'Extreme']:
                print(f"      {sev}: {sev_dist.get(sev, 0)}")

        # Size statistics
        if 'n_samples' in audit.columns:
            print(f"   Sample size range: {audit['n_samples'].min()} - {audit['n_samples'].max()}")
            print(f"   Median samples: {audit['n_samples'].median():.0f}")

        if 'imbalance_ratio' in audit.columns:
            print(f"   IR range: {audit['imbalance_ratio'].min():.1f} - {audit['imbalance_ratio'].max():.1f}")
            print(f"   Median IR: {audit['imbalance_ratio'].median():.1f}")

        if 'n_minority' in audit.columns:
            print(f"   Minority range: {audit['n_minority'].min()} - {audit['n_minority'].max()}")

        break

if not audit_found:
    print("   No dataset audit file found (run benchmark first)")

# ============================================================
# 6. RESULTS SUMMARY
# ============================================================
print("\n" + "="*60)
print("6. RESULTS SUMMARY")
print("="*60)

results_paths = [
    Path("nldf_results_41_v2_final/fold_level_results.csv"),
    Path("analysis_output/fold_level_results.csv"),
    Path("nldf_results_41_fixed/fold_level_results.csv"),
]

results_found = False
for rp in results_paths:
    if rp.exists():
        results = pd.read_csv(rp)
        results_found = True

        print(f"   Source: {rp}")
        print(f"   File size: {rp.stat().st_size / (1024*1024):.1f} MB")
        print(f"   Total experiments: {len(results)}")
        print(f"   Expected: 24,600")
        print(f"   Completeness: {len(results)/24600*100:.1f}%")
        print(f"   Datasets: {results['Dataset'].nunique()}/41")
        print(f"   Methods: {sorted(results['Method'].unique())}")

        # N-LDF check
        nldf_count = len(results[results['Method'] == 'N-LDF'])
        print(f"   N-LDF experiments: {nldf_count}")

        if len(results) > 0:
            # Timestamp range
            if 'timestamp' not in results.columns:
                print(f"   Date: Not recorded (add timestamp to results)")
            else:
                print(f"   Date range: {results['timestamp'].min()} to {results['timestamp'].max()}")

        break

if not results_found:
    print("   No results file found (run benchmark first)")

# ============================================================
# 7. FILES HASHES (for reproducibility)
# ============================================================
print("\n" + "="*60)
print("7. FILE INTEGRITY (SHA256)")
print("="*60)

code_files = [
    "nldf.py",
    "reproduce_paper_tables.py",
    "supplementary_analysis.py",
    "src/nldf_postprocessing.py",
]

for cf in code_files:
    fp = Path(cf)
    if fp.exists():
        sha = hashlib.sha256(fp.read_bytes()).hexdigest()[:16]
        print(f"   {cf:30s}: SHA256={sha}...")
    else:
        print(f"   {cf:30s}: Not found")

# ============================================================
# 8. ENVIRONMENT VARIABLES
# ============================================================
print("\n" + "="*60)
print("8. RELEVANT ENVIRONMENT VARIABLES")
print("="*60)

env_vars = [
    "PYTHONPATH", "CONDA_DEFAULT_ENV", "VIRTUAL_ENV",
    "CUDA_VISIBLE_DEVICES", "OMP_NUM_THREADS",
    "COLAB_GPU", "COLAB_TPU_ADDRESS",
]

for var in env_vars:
    value = os.environ.get(var, "NOT SET")
    if value != "NOT SET":
        print(f"   {var:25s}: {value}")

# ============================================================
# 9. REPRODUCIBILITY CHECKLIST
# ============================================================
print("\n" + "="*60)
print("9. REPRODUCIBILITY CHECKLIST")
print("="*60)

checklist = {
    "Random seed fixed": "✅" if benchmark_config["random_state"] == 42 else "❌",
    "Package versions recorded": "✅" if all(v != "NOT INSTALLED" for v in packages.values()) else "⚠️",
    "Dataset sources documented": "✅" if audit_found else "⚠️",
    "Hyperparameters specified": "✅ (nldf.build_sampler; see Cell 3)",
    "Code version controlled": "⚠️ (use git for tracking)",
    "Results archived": "✅" if results_found else "❌",
    "Environment documented": "✅",
    "GPU seed fixed": "⚠️ (XGBoost may use GPU randomness)",
}

for item, status in checklist.items():
    print(f"   {status} {item}")

# ============================================================
# 10. SAVE REPRODUCIBILITY REPORT
# ============================================================
print("\n" + "="*60)
print("10. SAVING REPORT")
print("="*60)

report = {
    "system_info": system_info,
    "hardware_info": {k: str(v) for k, v in hardware_info.items()},
    "package_versions": packages,
    "benchmark_config": benchmark_config,
    "dataset_info": {
        "total": len(audit) if audit_found else 41,
        "source": "KEEL imbalanced datasets repository",
        "url": "https://sci2s.ugr.es/keel/datasets.php",
    },
    "results_summary": {
        "total_experiments": len(results) if results_found else 0,
        "nldf_experiments": len(results[results['Method'] == 'N-LDF']) if results_found else 0,
        "completeness_pct": round(len(results)/24600*100, 1) if results_found else 0,
    } if results_found else {},
    "reproducibility_checklist": checklist,
    "generated_at": datetime.datetime.now().isoformat(),
    "generated_by": "Cell 10 - Reproducibility Report",
}

output_dir = Path("reproducibility_report")
output_dir.mkdir(exist_ok=True)

# Save as JSON
report_path = output_dir / "reproducibility_report.json"
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2, default=str)
print(f"   ✅ JSON report: {report_path}")

# Save as human-readable text
txt_path = output_dir / "reproducibility_report.txt"
with open(txt_path, 'w') as f:
    f.write("="*80 + "\n")
    f.write("COMPUTATIONAL ENVIRONMENT & REPRODUCIBILITY REPORT\n")
    f.write(f"Generated: {datetime.datetime.now().isoformat()}\n")
    f.write("="*80 + "\n\n")

    for section, data in report.items():
        if section != "generated_at" and section != "generated_by":
            f.write(f"\n{section.upper()}\n")
            f.write("-"*40 + "\n")
            if isinstance(data, dict):
                for k, v in data.items():
                    f.write(f"  {k}: {v}\n")
            elif isinstance(data, list):
                for item in data:
                    f.write(f"  - {item}\n")
            f.write("\n")

print(f"   ✅ Text report: {txt_path}")

# Also create requirements.txt for pip
req_path = output_dir / "requirements.txt"
with open(req_path, 'w') as f:
    f.write(f"# Generated: {datetime.datetime.now().isoformat()}\n")
    f.write(f"# Python: {sys.version.split()[0]}\n\n")
    for pkg, ver in sorted(packages.items()):
        if ver != "NOT INSTALLED" and pkg != "python":
            f.write(f"{pkg}=={ver}\n")
print(f"   ✅ requirements.txt: {req_path}")

# ============================================================
# FINAL SUMMARY
# ============================================================
print(f"\n{'='*80}")
print(f"📋 REPRODUCIBILITY REPORT COMPLETE")
print(f"{'='*80}")
print(f"📁 All files saved to: {output_dir}/")
print(f"   - reproducibility_report.json  (machine-readable)")
print(f"   - reproducibility_report.txt   (human-readable)")
print(f"   - requirements.txt             (pip install)")
print(f"\n💡 For the manuscript, include:")
print(f"   1. requirements.txt in supplementary materials")
print(f"   2. Reproducibility section referencing this report")
print(f"   3. Hardware specs in experimental setup")
print(f"{'='*80}")

# ============================================================
# PRINT CITABLE INFORMATION
# ============================================================
print(f"\n📝 CITABLE ENVIRONMENT SUMMARY:")
print(f"   All experiments conducted on {system_info['os']} {system_info['os_release']}")
print(f"   with Python {system_info['python_version'].split()[0]}, ")
print(f"   scikit-learn {packages['scikit-learn']}, ")
print(f"   XGBoost {packages['xgboost']}, ")
print(f"   and imbalanced-learn {packages['imbalanced-learn']}. ")
if hardware_info.get('gpu_name'):
    print(f"   GPU: {hardware_info['gpu_name']} ({hardware_info.get('gpu_memory_mb', 'N/A')}). ")
print(f"   Random seed fixed at {benchmark_config['random_state']}.")
print(f"   Complete environment specifications available in reproducibility_report/.")

### Cell 10: environment summary for the Methods section

In [ ]:
"""Cell 10: QUICK REPRODUCIBILITY CHECK - environment summary for the Methods section"""

import datetime
import importlib.metadata
import platform
import sys
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb

print("="*70)
print("📋 QUICK REPRODUCIBILITY CHECK - Copy to Methods Section")
print("="*70)

# Detect environment
is_colab = "google.colab" in sys.modules

# Get versions
versions = {
    "Python": sys.version.split()[0],
    "NumPy": np.__version__,
    "Pandas": pd.__version__,
    "Scikit-learn": sklearn.__version__,
    "XGBoost": xgb.__version__,
}

for pkg in ["imbalanced-learn", "scikit-posthocs", "scipy"]:
    try:
        versions[pkg] = importlib.metadata.version(pkg)
    except:
        versions[pkg] = "N/A"

# Hardware
try:
    import psutil
    ram_gb = round(psutil.virtual_memory().total / (1024**3))
    cpu_count = psutil.cpu_count()
except:
    ram_gb = "N/A"
    cpu_count = "N/A"

gpu_info = ""
try:
    result = !nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null
    if result and result[0]:
        gpu_info = f", {result[0].strip()}"
except:
    pass

# Print methods section template
print(f"""
METHODS SECTION TEMPLATE:
-------------------------
All experiments were conducted on a {platform.system()} system
({'Google Colab with ' if is_colab else ''}{cpu_count} CPU cores, {ram_gb} GB RAM{gpu_info})
using Python {versions['Python']}.

Machine learning models were implemented using
scikit-learn {versions['Scikit-learn']} and XGBoost {versions['XGBoost']}.
Imbalanced learning methods were implemented using
imbalanced-learn {versions['imbalanced-learn']}.

Our proposed N-LDF method (nldf.py, SHA-256 {NLDF_SHA256[:8]}...) was implemented from scratch
with the following key components:
(1) class-prior-corrected local density estimation (λ=0.7),
(2) adaptive kernel with local bandwidth scaling,
(3) quantile-based boundary that widens with imbalance ratio,
(4) soft seed weighting replacing hard categorization,
(5) safe-directed neighbor selection,
(6) local covariance-based jitter generation,
(7) three-stage acceptance filtering, and
(8) a fixed sampling ratio of 0.75 (the inner-CV option of the class was disabled).

All experiments used a fixed random seed (42) with
5-fold cross-validation repeated 5 times across
41 KEEL imbalanced benchmark datasets.
Statistical significance was assessed using
Friedman test with Holm-corrected Wilcoxon signed-rank post-hoc tests.

Complete environment specifications and reproducibility
details are available in the supplementary materials.
""")

# Save for paper
with open("methods_section_template.txt", "w") as f:
    f.write(f"Generated: {datetime.datetime.now().isoformat()}\n")
    f.write(f"Environment: {'Google Colab' if is_colab else 'Local'}\n")
    f.write(f"Python: {versions['Python']}\n")
    f.write(f"scikit-learn: {versions['Scikit-learn']}\n")
    f.write(f"XGBoost: {versions['XGBoost']}\n")
    f.write(f"imbalanced-learn: {versions['imbalanced-learn']}\n")

print("\n✅ Template saved to: methods_section_template.txt")